# 00 - Data Preprocessing
**CSC61304 Group 6 - Machine Learning for Disaster Preparedness: Rainfall-Driven Flood Risk in Nepal**

Shared base notebook. Loads the CHIRPS rainfall dataset, cleans it, engineers
features, creates the flood risk label, and produces the train/test splits
used by all five models in this project.

Pipeline: **Load -> Clean -> Engineer Features -> Create Target Label ->
Handle Missing Values -> Encode -> Scale -> Split -> Save**


In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

RAW_PATH = "../dataset/raw.csv" if os.path.exists("../dataset/raw.csv") else "dataset/raw.csv"
DATASET_DIR = os.path.dirname(RAW_PATH)
PROCESSED_DIR = os.path.join(DATASET_DIR, "processed")
RAW_DIR = os.path.join(DATASET_DIR, "raw")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Step 1-2: Load & Clean

Loads the raw CHIRPS CSV (row index 1 holds HXL tags and is skipped).
Filters to district-level records only, using the `adm_level` whose number
of distinct `PCODE`s matches Nepal's 77 districts.


In [2]:
RAW_PATH = "../dataset/raw.csv" if os.path.exists("../dataset/raw.csv") else "dataset/raw.csv"

# Auto-detect if HXL tag row exists in raw dataset
df_check = pd.read_csv(RAW_PATH, nrows=2)
if df_check.columns[0].startswith('#') or str(df_check.iloc[0, 0]).startswith('#'):
    df = pd.read_csv(RAW_PATH, skiprows=[1])
else:
    df = pd.read_csv(RAW_PATH)

df.columns = [c.strip().lower() for c in df.columns]

if 'version' in df.columns:
    df = df.drop(columns=['version'])

df['date'] = pd.to_datetime(df['date'])

numeric_cols = ['rfh', 'r1h', 'r3h', 'rfh_avg', 'r1h_avg', 'r3h_avg',
                'rfq', 'r1q', 'r3q', 'n_pixels']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

level_counts = df.groupby('adm_level')['pcode'].nunique()
print("Unique pcodes per adm_level:\n", level_counts)

district_level = (level_counts - 77).abs().idxmin()
print(f"\nSelected adm_level = {district_level} as district level "
      f"({level_counts[district_level]} unique pcodes)")

df = df[df['adm_level'] == district_level].copy()
df = df.dropna(subset=['rfh']).drop_duplicates()

print(df.shape)
df.head()

Unique pcodes per adm_level:
 adm_level
1     7
2    77
Name: pcode, dtype: int64

Selected adm_level = 2 as district level (77 unique pcodes)
(126203, 14)


,date,adm_level,adm_id,pcode,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q
11473,1981-01-01,2,1008005,NP0769,63.0,11.634921,10.846561,NaN,NaN,NaN,NaN,104.974976,NaN,NaN
11474,1981-01-11,2,1008005,NP0769,63.0,3.523809,11.660317,NaN,NaN,NaN,NaN,51.162350,NaN,NaN
11475,1981-01-21,2,1008005,NP0769,63.0,33.904762,11.483598,49.063490,33.990475,NaN,NaN,236.021070,138.65820,NaN
11476,1981-02-01,2,1008005,NP0769,63.0,8.015873,12.115344,45.444447,35.259260,NaN,NaN,76.047970,125.29900,NaN
11477,1981-02-11,2,1008005,NP0769,63.0,1.095238,15.313757,43.015873,38.912697,NaN,NaN,30.005470,109.34395,NaN


## Step 3: Feature Engineering

Derives `year`, `month`, and `decade_num` (which 10-day period of the month)
from `date`. `district_zone` is derived from the province-code prefix of
`PCODE`, mapped to Terai/Hill/Mountain.


In [3]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['decade_num'] = df['date'].dt.day.apply(lambda d: 1 if d <= 10 else (2 if d <= 20 else 3))

df['province_code'] = df['pcode'].astype(str).str[:4].str.upper()

province_zone_map = {
    'NP01': 'Hill',      # Koshi
    'NP02': 'Terai',     # Madhesh
    'NP03': 'Hill',      # Bagmati
    'NP04': 'Mountain',  # Gandaki
    'NP05': 'Terai',     # Lumbini
    'NP06': 'Mountain',  # Karnali
    'NP07': 'Hill',      # Sudurpashchim
}
df['district_zone'] = df['province_code'].map(province_zone_map)

print("Unmapped province codes (should be empty):")
print(df.loc[df['district_zone'].isna(), 'province_code'].unique())

df[['pcode', 'province_code', 'district_zone']].drop_duplicates().head(10)

Unmapped province codes (should be empty):
<StringArray>
[]
Length: 0, dtype: str


,pcode,province_code,district_zone
11473,NP0769,NP07,Hill
13112,NP0550,NP05,Terai
14751,NP0443,NP04,Mountain
16390,NP0774,NP07,Hill
18029,NP0767,NP07,Hill
19668,NP0768,NP07,Hill
21307,NP0557,NP05,Terai
22946,NP0233,NP02,Terai
24585,NP0558,NP05,Terai
26224,NP0326,NP03,Hill


## Step 4: Target Label Creation

Buckets `rfq` (rainfall anomaly %) into `flood_risk_level` using percentiles
of the observed distribution: Low, Moderate, High, and Extreme.


In [4]:
q75, q90, q97 = df['rfq'].quantile([0.75, 0.90, 0.97])

def flood_risk_percentile(rfq):
    if rfq >= q97:
        return 'Extreme'
    elif rfq >= q90:
        return 'High'
    elif rfq >= q75:
        return 'Moderate'
    else:
        return 'Low'

df['flood_risk_level'] = df['rfq'].apply(flood_risk_percentile)

df['flood_risk_level'].value_counts(normalize=True)

flood_risk_level
Low         0.749998
Moderate    0.149996
High        0.069998
Extreme     0.030007
Name: proportion, dtype: float64

## Step 5: Missing Value Handling

Fills missing numeric values using each district's own median (grouped by
`PCODE`), since rainfall patterns differ considerably between districts.


In [5]:
for col in numeric_cols:
    df[col] = df.groupby('pcode')[col].transform(lambda x: x.fillna(x.median()))

df = df.dropna(subset=['rfq', 'flood_risk_level'])

df[numeric_cols].isna().sum()

rfh         0
r1h         0
r3h         0
rfh_avg     0
r1h_avg     0
r3h_avg     0
rfq         0
r1q         0
r3q         0
n_pixels    0
dtype: int64

## Step 6-7: Encoding + Scaling

Label-encodes `district_zone` and `flood_risk_level`, then builds two
feature sets:

- **Classification features** — used by Logistic Regression / Decision
  Tree, Naive Bayes, Random Forest, and K-Means.
- **Regression features** — used by Linear Regression, with its own
  separately-fit scaler.


In [6]:
le_zone = LabelEncoder()
df['district_zone_enc'] = le_zone.fit_transform(df['district_zone'])

le_label = LabelEncoder()
df['flood_risk_level_enc'] = le_label.fit_transform(df['flood_risk_level'])

clf_feature_cols = ['rfh', 'r1h', 'r3h', 'rfh_avg', 'r1h_avg', 'r3h_avg',
                     'r1q', 'r3q', 'n_pixels',
                     'year', 'month', 'decade_num', 'district_zone_enc']
clf_feature_cols = [c for c in clf_feature_cols if c in df.columns]

clf_scaler = StandardScaler()
df_clf_scaled = df.copy()
df_clf_scaled[clf_feature_cols] = clf_scaler.fit_transform(df[clf_feature_cols])

reg_feature_cols = [c for c in clf_feature_cols if c != 'rfh']

reg_scaler = StandardScaler()
X_reg_scaled = df.copy()
X_reg_scaled[reg_feature_cols] = reg_scaler.fit_transform(df[reg_feature_cols])

df_clf_scaled[clf_feature_cols].describe()

,rfh,r1h,r3h,rfh_avg,r1h_avg,r3h_avg,r1q,r3q,n_pixels,year,month,decade_num,district_zone_enc
count,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05,1.262030e+05
mean,-1.080990e-17,-3.603301e-17,3.963631e-17,-9.728912e-17,2.161980e-17,-7.206601e-18,-2.549335e-16,5.630157e-17,1.585452e-16,7.064271e-15,-2.026857e-18,-2.100664e-17,-8.647922e-17
std,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00
min,-7.648793e-01,-8.441040e-01,-9.814419e-01,-8.848830e-01,-8.997343e-01,-1.002105e+00,-2.050540e+00,-2.479322e+00,-1.566977e+00,-1.694013e+00,-1.584343e+00,-1.223811e+00,-9.578133e-01
25%,-6.894884e-01,-7.416801e-01,-8.185997e-01,-7.540725e-01,-7.595398e-01,-8.310952e-01,-6.251937e-01,-6.529316e-01,-5.536449e-01,-8.571489e-01,-1.004779e+00,-1.223811e+00,-9.578133e-01
50%,-4.844717e-01,-4.914738e-01,-4.318570e-01,-5.288285e-01,-5.398228e-01,-4.520677e-01,-1.700961e-01,-1.231120e-01,-2.237227e-01,-2.028453e-02,-1.354321e-01,7.471375e-04,2.512297e-01
75%,3.946989e-01,5.290453e-01,6.587978e-01,5.948325e-01,6.033313e-01,7.310606e-01,3.972174e-01,4.729200e-01,3.418581e-01,8.926585e-01,7.339147e-01,1.225306e+00,1.460273e+00
max,1.135419e+01,7.930022e+00,4.881004e+00,4.026437e+00,3.825985e+00,3.320738e+00,1.064191e+01,1.118345e+01,5.219993e+00,1.729523e+00,1.603261e+00,1.225306e+00,1.460273e+00


## Step 8: Train/Test Split + Save

Classification split is stratified on `flood_risk_level_enc`. Regression
target (`y_reg`) is kept in raw mm so RMSE/MAE report in interpretable
rainfall units.


In [7]:
X = df_clf_scaled[clf_feature_cols]
y_class = df_clf_scaled['flood_risk_level_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

X_reg = X_reg_scaled[reg_feature_cols]
y_reg = df['rfh']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

df.to_csv(os.path.join(PROCESSED_DIR, "nepal_flood_processed.csv"), index=False)

X_train.to_csv(os.path.join(PROCESSED_DIR, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(PROCESSED_DIR, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(PROCESSED_DIR, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(PROCESSED_DIR, "y_test.csv"), index=False)

X_train_reg.to_csv(os.path.join(PROCESSED_DIR, "X_train_reg.csv"), index=False)
X_test_reg.to_csv(os.path.join(PROCESSED_DIR, "X_test_reg.csv"), index=False)
y_train_reg.to_csv(os.path.join(PROCESSED_DIR, "y_train_reg.csv"), index=False)
y_test_reg.to_csv(os.path.join(PROCESSED_DIR, "y_test_reg.csv"), index=False)

print(f"Saved processed data and both splits to {PROCESSED_DIR}/")
print(f"\nClassification features ({len(clf_feature_cols)}): {clf_feature_cols}")
print(f"Regression features ({len(reg_feature_cols)}): {reg_feature_cols}")

Saved processed data and both splits to ../dataset\processed/

Classification features (13): ['rfh', 'r1h', 'r3h', 'rfh_avg', 'r1h_avg', 'r3h_avg', 'r1q', 'r3q', 'n_pixels', 'year', 'month', 'decade_num', 'district_zone_enc']
Regression features (12): ['r1h', 'r3h', 'rfh_avg', 'r1h_avg', 'r3h_avg', 'r1q', 'r3q', 'n_pixels', 'year', 'month', 'decade_num', 'district_zone_enc']
